In [1]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
from shutil import move

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [3]:
form_classified = pd.read_csv("./csv/form_classified.csv", encoding="utf-8")
form_emestrada = pd.read_csv("./csv/form_emestrada.csv", encoding="utf-8")

In [4]:
form_emestrada.head()

,year,exam_name,statement
0,2010,química - 2010 - junio,a) Nitrito de plata b) Hidróxido de magnesio c...
1,2010,química - 2010 - junio,a) Peróxido de rubidio b) Hidrogenocarbonato d...
2,2010,química - 2010 - reserva 1,a) Permanganato de bario b) Peróxido de potasi...
3,2010,química - 2010 - reserva 1,a) Hidrogenocarbonato de sodio b) Hidróxido de...
4,2010,química - 2010 - reserva 2,a) Pentafluoruro de antimonio b) Óxido de plom...


In [5]:
form_classified.sample(5).statement

130    A1. Formule o nombre los siguientes compuestos...
29     1.- Formule o nombre los compuestos siguientes...
1      1.- Formule o nombre los compuestos siguientes...
187    4A. Nombre o formule los siguientes compuestos...
167    A2. Formule o nombre los siguientes compuestos...
Name: statement, dtype: str

### Formulación 2025

In [6]:
for i in form_classified.loc[form_classified.year == 2025, "statement"]:
    print(i)

4A. Nombre o formule los siguientes compuestos:
a) CH₂=CH―CH₂OH b) CH₃―C(CH₃)₂―C == C H c) Propanamida d) Ácido benzoico
4A. Nombre o formule los siguientes compuestos:
O
―
a) CH₃―CH₂―CH₂―CH₂―C―NH₂ b) C H₃― ―OH c) Butanodial d) 6-Bromohex-2-ino
4A. Nombre o formule los siguientes compuestos:
a) CH₂=CH―CH₂―NH₂ b) CH₃―O―CH₃ c) Etilbenceno d) Difenil éter
4A. Nombre o formule los siguientes compuestos:
a) CH₃―C=CH―CH₃ b) CH₃―CH₂―CH(OH)―CH₃ c) Etanoato de etilo d) Metil propil éter
―
CH₃
4A. Nombre o formule los siguientes compuestos:
a) CH₃―CH=CH―CH=CH₂ b) CH₃―CH(OH)―CH₂―CN c) 3–Metilpent–2–eno d) Etanamida
4A. Nombre o formule los siguientes compuestos:
a) CH₃―CH₂―O―CH₃ b) CH₃―CH―CH₂―CHO c) Etanamina d) Penta–1,3–dieno
―
CH₃


### Extraer nombramiento fórmulas orgánicas 2025

In [7]:
def extraer_nombrar_compuestos_25(statement):
    if not pd.isna(statement):
        start_position = statement.find("c)")
        end_position = statement.find("d) ")
        statement = statement[start_position:end_position]
        statement = statement.replace(";", "").replace("\n", " ").strip()
    
    return statement

In [8]:
form_classified[form_classified.year == 2025].statement.apply(extraer_nombrar_compuestos_25)

182          c) Propanamida
183           c) Butanodial
184          c) Etilbenceno
185    c) Etanoato de etilo
186    c) 3–Metilpent–2–eno
187            c) Etanamina
Name: statement, dtype: str

In [9]:
form_emestrada[form_emestrada.year == 2025].statement.apply(extraer_nombrar_compuestos_25)

178    c) 3-Metilpent-2-eno
179          c) Propanamida
180          c) Etilbenceno
181            c) Etanamina
Name: statement, dtype: str

In [10]:
mask = form_classified.year == 2025
form_classified.loc[mask, "statement"] = form_classified.loc[mask].statement.apply(extraer_nombrar_compuestos_25)

In [11]:
mask = form_emestrada.year == 2025
form_emestrada.loc[mask, "statement"] = form_emestrada.loc[mask].statement.apply(extraer_nombrar_compuestos_25)

## Extraer apartados de nombrar en ejercicios de fórmulación (**previos a 2025**)

In [12]:
def extraer_nombrar_compuestos(statement):
    if not pd.isna(statement):
        start_position = statement.find("a)")
        if " d) " in statement:
            end_position = statement.find(" d) ")
        else:
            end_position = -1
        statement = statement[start_position:end_position]
        # unir valencia con Stock
        statement = re.sub(r"\s+(\([IVX]+\))", r"\1", statement)
        statement = statement.replace(";", "").replace("\n", " ").strip()
        #statement = re.sub(r"[^0-9a-záéíóú]", "", statement.lower())
    
    return statement

In [13]:
for i in form_classified.statement.sample(5).apply(extraer_nombrar_compuestos):
    print(i)

a) Peróxido de calcio b) Sulfato de zinc c) Butanamida d) Ag₂S e) HBrO₄ f) CH₃CHBrCHBrCH
a) Cloruro de amonio b) Ácido nitroso c) Etanoato de propilo d) CaO₂ e) Hg(BrO₂)₂ f) CH₃CHFCH₂CH₃
a) Peróxido de rubidio b) Hidrogenocarbonato de sodio c) Ciclohexanona
a) Peróxido de sodio b) Sulfuro de amonio c) p-Metilfenol
a) Dióxido de titanio b) Sulfato de amonio c) Ácido 2-bromobutanoico


In [14]:
for i in form_emestrada.statement.sample(5):
    print(i)

a) Bromato de berilio b) Sulfuro de amonio c) 4-Bromo-5-etiloctano
a) Ácido hipocloroso b) Sulfuro de cadmio c) Permanganato de potasio
a) Sulfuro de aluminio b) Ácido peryódico c) Etanoato de propilo
a) Cromato de plata b) Dióxido de estaño c) Ácido nitroso
a) Peróxido de hidrógeno b) Hidrogenosulfito de sodio c) Hidruro de boro


In [15]:
mask = form_classified.year != 2025
form_classified.loc[mask, "statement"] = form_classified.loc[mask].statement.apply(extraer_nombrar_compuestos)

## Combinar los datasets
Hacer *match* entre los ejercicios de formulación de *emestrada* y los clasificados, usando como clave el enunciado de formulación. 

In [16]:
# x: classified, y: emestrada
form_merged = pd.merge(form_classified, form_emestrada, on=["statement", "year"])

In [17]:
form_merged.shape

(145, 9)

In [18]:
form_merged.columns

Index(['year', 'exam_name_x', 'exercise_number', 'topic', 'exercise_type',
       'statement', 'exam_option', 'exam_file_path', 'exam_name_y'],
      dtype='str')

In [19]:
form_merged.year.aggregate(["min", "max"])

min    2010
max    2025
Name: year, dtype: int64

In [20]:
cols_to_keep = ["year", "exam_file_path", "exam_name_y"]
form_merged = form_merged.drop([col for col in form_merged.columns if col not in cols_to_keep], axis = 1)

In [21]:
form_merged = form_merged.rename(columns = {"exam_name_y" : "filename_emestrada",
                     "exam_file_path" : "source_file_path",})

In [22]:
form_merged.head()

,year,source_file_path,filename_emestrada
0,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - junio
1,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 1
2,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 1
3,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 2
4,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - septiembre


In [23]:
form_merged.tail()

,year,source_file_path,filename_emestrada
140,2024,examenes_quimica/2024_quimica/2024_quimica_QUÍ...,química - 2024 - julio
141,2024,examenes_quimica/2024_quimica/2024_quimica_QUÍ...,química - 2024 - julio
142,2025,examenes_quimica/2025_quimica/2025_quimica_Qui...,química - 2025 - reserva 1
143,2025,examenes_quimica/2025_quimica/2025_quimica_Qui...,química - 2025 - reserva 2
144,2025,examenes_quimica/2025_quimica/2025_quimica_Qui...,química - 2025 - julio


In [24]:
form_merged = form_merged.drop_duplicates()

In [25]:
form_merged.year.unique()

array([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020,
       2021, 2022, 2023, 2024, 2025])

In [26]:
form_emestrada[form_emestrada.year == 2023].statement

154    a) Óxido de manganeso(VII) b) Dicromato de pot...
155    a) Selenuro de plata b) Ácido clórico c) 1,3,5...
156    a) Sulfuro de aluminio b) Ácido peryódico c) E...
157    a) Hidruro de estroncio b) Hidróxido de bario ...
158    a) Hidróxido de plomo(IV) b) Nitrato de calcio...
159    a) Dióxido de titanio b) Ácido hipobromoso c) ...
160    a) Peróxido de sodio b) Sulfuro de amonio c) p...
161    a) Hidróxido de hierro(III) b) Ácido nitroso c...
162    a) Telururo de hidrogeno b) Sulfato de amonio ...
163    a) Peróxido de bario b) Bromuro de calcio c) P...
164    a) Óxido de vanadio(V) b) Hidruro de plomo(IV)...
165    a) Peróxido de rubidio b) Hidrogenocarbonato d...
Name: statement, dtype: str

In [27]:
form_classified[form_classified.year == 2023].statement

158    a) Sulfuro de aluminio b) Ácido peryódico c) E...
159    a) Hidruro de estroncio b) Hidróxido de bario ...
160    a) Hidróxido de plomo(IV) b) Nitrato de calcio...
161    a) Dióxido de titanio b) Ácido hipobromoso c) ...
162    a) Peróxido de sodio b) Sulfuro de amonio c) p...
163    a) Hidróxido de hierro(III) b) Ácido nitroso c...
164    a) Telururo de hidrógeno b) Sulfato de amonio ...
165    a) Peróxido de bario b) Bromuro de calcio c) P...
166    a) Óxido de manganeso(VII) b) Dicromato de pot...
167    a) Selenuro de plata b) Ácido clórico c) 1,3,5...
168    a) Óxido de vanadio(V) b) Hidruro de plomo(IV)...
169    a) Peróxido de rubidio b) Hidrogenocarbonato d...
Name: statement, dtype: str

In [28]:
form_classified[form_classified.year == 2023].statement

158    a) Sulfuro de aluminio b) Ácido peryódico c) E...
159    a) Hidruro de estroncio b) Hidróxido de bario ...
160    a) Hidróxido de plomo(IV) b) Nitrato de calcio...
161    a) Dióxido de titanio b) Ácido hipobromoso c) ...
162    a) Peróxido de sodio b) Sulfuro de amonio c) p...
163    a) Hidróxido de hierro(III) b) Ácido nitroso c...
164    a) Telururo de hidrógeno b) Sulfato de amonio ...
165    a) Peróxido de bario b) Bromuro de calcio c) P...
166    a) Óxido de manganeso(VII) b) Dicromato de pot...
167    a) Selenuro de plata b) Ácido clórico c) 1,3,5...
168    a) Óxido de vanadio(V) b) Hidruro de plomo(IV)...
169    a) Peróxido de rubidio b) Hidrogenocarbonato d...
Name: statement, dtype: str

## Renombrar archivos

In [29]:
form_merged[form_merged.year == 2020].filename_emestrada

92     química - 2020 - reserva 1
94     química - 2020 - reserva 2
95         química - 2020 - junio
96     química - 2020 - reserva 3
97     química - 2020 - reserva 4
98    química - 2020 - septiembre
Name: filename_emestrada, dtype: str

In [30]:
for idx, row in form_merged.iterrows():
    old = ROOT / row.source_file_path
    new = old.parent / f"{row.filename_emestrada}.pdf"

    #print(f"{old} --> {new}")
    try:
        move(old, new)
    except Exception as e:
        print(f"Archivo: {old.name}. Error: {e}")